<a href="https://colab.research.google.com/github/pop123-ux/doc-assistant-hf/blob/main/coded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook represents the pipeline of building an app where users upload PDFs, Word files, or text, and receive concise summaries or answers to specific questions about the document

- facebook/bart-large-cnn for summarization and deepset/roberta-base-squad2 for question answering

- Toggle to switch between abstractive (rewriting) and extractive (bullet points) summarization

In [ ]:
!hf auth login

Hint: A new version of huggingface_hub (1.26.0) is available! You are using version 1.23.0.
To update, run: hf update
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: G9L6-VG4E

    Waiting for authorization.....
Token is valid.
The token `oauth-pop123ux` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-pop123ux`
Note: This token will be refreshed automatically when it expires.


In [ ]:
import json
import os
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import hf_hub_download
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

dataset1 = load_dataset('knkarthick/samsum')


README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
dataset1

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [ ]:
model1 = "facebook/bart-large-cnn"

tokenizer1 = AutoTokenizer.from_pretrained(model1, use_fast=True)
tokenizer1.pad_token = tokenizer1.eos_token
tokenizer1.padding_side = "right"

In [ ]:
# We must import a custom tokenizer chat template since roberta does not come with one

tokenizer1.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
)

def tokenize(example):
  messages = [
      {'role':'system', 'content':"You are an expert AI conversational summarizer."},
      {'role':'user', 'content':example['dialogue']},
  ]
  full_text = tokenizer1.apply_chat_template(messages, tokenize=False)

  # We apply the tokenizer function with a max_length = 512
  tokenized = tokenizer1(full_text, truncation=True, max_length=512)

  return tokenized

In [ ]:
tokenized_dataset = dataset1.map(tokenize, batched=False)
tokenized_dataset

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask'],
        num_rows: 819
    })
})

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 12.7 MB/s eta 0:00:00


In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model

# Configure the 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the model with quantization
model = AutoModelForSeq2SeqLM.from_pretrained(
    model1,
    quantization_config=bnb_config,
    device_map="auto",
)
model1

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

'facebook/bart-large-cnn'

In [ ]:
# Configure LoRA (PEFT)
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=8,
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model1 = get_peft_model(model, peft_config)
model1.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,179,648 || all params: 407,470,080 || trainable%: 0.2895


Now as we're done with the first model we're going to continue with the second one, which is a roberta-base pretrained on the squad2 dataset

In [ ]:
model2 = "deepset/roberta-base-squad2"

In [ ]:
tokenizer2 = AutoTokenizer.from_pretrained(model2)

model2 = AutoModelForCausalLM.from_pretrained(model2, is_decoder=True)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [ ]:
!pip install -q pypdf python-docx gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.7 MB/s eta 0:00:00


Now we're going to implement the gradio interface and the document parsing functions

In [ ]:
import gradio as gr
import os
from pypdf import PdfReader
from docx import Document
from transformers import pipeline

# Load the models using HF Pipelines
summarizer = pipeline("text-generation", model=model1, tokenizer=tokenizer1, device_map='auto')

qa = pipeline('text-generation', model=model2, tokenizer=tokenizer2, device_map='auto')


def extract_text_from_file(file):
    if file is None:
        return ""

    file_path = file.name
    ext = os.path.splitext(file_path)[1].lower()
    extracted_text = ""

    try:
        if ext == ".pdf":
            reader = PdfReader(file_path)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
        elif ext in [".docx", ".doc"]:
            doc = Document(file_path)
            for para in doc.paragraphs:
                text = para.text
                if text:
                    extracted_text += text + "\n"
        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                extracted_text = f.read()
        else:
            return "Unsupported file format. Please upload PDF, DOCX, or TXT"

        return extracted_text.strip()

    except Exception as e:
        return f"Error reading file: {str(e)}"

def process_summary(text, file):
    source_text = extract_text_from_file(file) if file else text
    if not source_text.strip():
        return "Please enter text or upload a file."

    # Quick fix for ultra-long docs to avoid pipeline crash
    input_text = source_text[:4000]
    prompt = f"Summarize the following text short and clear:\n{input_text}\nSummary:"

    outputs = summarizer(prompt, max_new_tokens=150, do_sample=False)
    return outputs[0]["generated_text"].split("Summary:")[-1].strip()

def process_qa(text, file, question):
    source_text = extract_text_from_file(file) if file else text

    if not source_text:
        return "Please provide a document or text to answer the question."

    if not question:
        return "Please enter a question to answer."

    prompt = f"Context: {source_text}\nQuestion: {question}\nAnswer:"

    result = qa(prompt, max_new_tokens=30)
    return result[0]["generated_text"].split("Answer:")[-1].strip()



Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED |  | 
mod

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
[transformers] RobertaForCausalLM LOAD REPORT from: deepset/roberta-base-squad2
Key                       | Status     | 
--------------------------+------------+-
qa_outputs.bias           | UNEXPECTED | 
qa_outputs.weight         | UNEXPECTED | 
lm_head.decoder.bias      | MISSING    | 
lm_head.dense.weight      | MISSING    | 
lm_head.layer_norm.weight | MISSING    | 
lm_head.layer_norm.bias   | MISSING    | 
lm_head.bias              | MISSING    | 
lm_head.dense.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
  gr.Markdown("# 📄 Multi-Format Document Assistant")
  gr.Markdown("AI-powered portofolio tool to summarize text and extract precise answers from documents.")

  with gr.Tab("Document Summarizer"):
    with gr.Row():
      with gr.Column():
        sum_text = gr.Textbox(label="Option A: Paste text directly", lines=6, placeholder="Enter text...")
        sum_file = gr.File(label="Option B: Upload Document (PDF, DOCX, TXT)", file_types=[".pdf", ".docx", ".txt"])
        summary_btn = gr.Button("Generate Summary", variant="primary")
      with gr.Column():
        summary_output = gr.Textbox(label='Summary Output', lines=6, interactive=False)

    summary_btn.click(fn=process_summary, inputs=[sum_text, sum_file], outputs=summary_output)

  with gr.Tab("Document QA System"):
    with gr.Row():
      with gr.Column():
        qa_text = gr.Textbox(label="Option A: Paste context directly", lines=6, placeholder="Enter context...")
        qa_file = gr.File(label="Option B: Upload Document (PDF, DOCX, TXT)", file_types=[".pdf", ".docx", ".txt"])
        qa_question = gr.Textbox(label="Your Question", lines=2, placeholder="What is the main revenue driver mentioned?")
        qa_btn = gr.Button("Find Answer", variant="primary")
      with gr.Column():
        qa_output = gr.Textbox(label='Extracted Answer', lines=4, interactive=False)

    qa_btn.click(fn=process_qa, inputs=[qa_text, qa_file, qa_question], outputs=qa_output)

# Launch the app with a public share link
demo.launch(share=True)


KeyboardInterrupt: 